In [ ]:
import Pkg
Pkg.activate(".")
Pkg.instantiate()
using CuLAMP
using LinearAlgebra
using Random


rng = Xoshiro(0) # Random number generator

N = 100
# Make a small, random OT problem for demonstration purposes
r, c, W, optimal_value = CuLAMP.generate_random_ot(N, N, rng)
args = EOTArgs(eta_p=1e-5, verbose=false) # define an argument struct to hold algorithm options

# First, run with Sinkhorn
plan1, φ, ψ = sinkhorn_log(r, c, W, args)
plan1_rounded = CuLAMP.round(plan1, r, c) # Round according to Algorithm 2 of Altschuler et al. 2017
SK_infeas = norm(c-sum(plan1', dims=2))+norm(r-sum(plan1, dims=2))

# Next, run with LAMP
plan2, ν = CuLAMP.LAMP(r, c, W, args)
plan2_rounded = CuLAMP.round(plan2, r, c) # Round, though probably not needed
LAMP_infeas = norm(c-sum(plan2', dims=2)) # no need for r marginal due to reparameterization

println("Sinkhorn Infeasibility: $SK_infeas, LAMP Infeasibility: $LAMP_infeas")
println("Sinkhorn Rounded Gap: $(dot(plan1_rounded, W)-optimal_value), LAMP Rounded Gap: $(dot(plan2_rounded, W)-optimal_value)")